# Drop Zone Defect Detector — 2-Stage Pipeline

## Stage 1 — Detection (EfficientNet B0)
"Is this photo the inside of an ice bin / drop zone?"
→ NO  → NOISE (skip)
→ YES → Stage 2

## Stage 2 — Quality (DINOv2 + Isolation Forest)
"Is this drop zone GOOD or a manufacturing DEFECT?"
→ GOOD / DEFECT / UNCERTAIN

## Drive folder structure
```
MyDrive/
  drop_zone_training/
    good/           ← normal drop zone photos
    bad/            ← manufacturing defect photos (caulking, rough edges, assembly)
    not_drop_zone/  ← any photo that is NOT the inside of the bin
                      (cube separator photos, exterior, data plates, etc.)
    trained_models/ ← both models saved here
```

## Steps
1. Runtime → Change runtime type → **T4 GPU**
2. Add photos to all three Drive folders
3. Run all cells top to bottom
4. Download both model files at the end

In [ ]:
# Cell 1 — Imports
import pickle, shutil, random
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

IMG_SIZE = 224  # EfficientNet B0

INFERENCE_TF = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

DINO_TF = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
# Cell 2 — Mount Google Drive and load photo folders
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE  = '/content/drive/MyDrive/drop_zone_training'
GOOD_DIR    = Path(DRIVE_BASE) / 'good'
BAD_DIR     = Path(DRIVE_BASE) / 'bad'
NOT_DZ_DIR  = Path(DRIVE_BASE) / 'not_drop_zone'
MODEL_OUT   = Path(DRIVE_BASE) / 'trained_models'
MODEL_OUT.mkdir(parents=True, exist_ok=True)

def load_images(folder: Path, label: str) -> list[Path]:
    if not folder.exists():
        raise FileNotFoundError(f'Folder not found: {folder}')
    paths = [p for p in folder.rglob('*')
             if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    print(f'  {label}: {len(paths)} photos')
    return paths

print(f'Loading from: {DRIVE_BASE}\n')
good_paths   = load_images(GOOD_DIR,   'good (drop zone — normal)')
bad_paths    = load_images(BAD_DIR,    'bad  (drop zone — defect)')
not_dz_paths = load_images(NOT_DZ_DIR, 'not_drop_zone')

if len(good_paths) < 10:
    raise RuntimeError(f'Only {len(good_paths)} good photos — need at least 10')
if len(bad_paths) == 0:
    raise RuntimeError('No bad photos found')
if len(not_dz_paths) < 10:
    raise RuntimeError(f'Only {len(not_dz_paths)} not_drop_zone photos — add more')

print(f'\nTotal positives (Stage 1): {len(good_paths) + len(bad_paths)}')
print(f'Total negatives (Stage 1): {len(not_dz_paths)}')

In [ ]:
# Cell 3 — Build Stage 1 detection dataset
# Positive: good + bad drop zone photos (both ARE drop zone photos)
# Negative: not_drop_zone photos
DETECT_ROOT = Path('dz_detect_split')
for split in ('train', 'val'):
    for cls in ('drop_zone', 'not_drop_zone'):
        (DETECT_ROOT / split / cls).mkdir(parents=True, exist_ok=True)

random.seed(42)

pos_paths = good_paths + bad_paths
random.shuffle(pos_paths)
split_pos = max(1, int(len(pos_paths) * 0.85))
for p in pos_paths[:split_pos]:
    shutil.copy(p, DETECT_ROOT / 'train' / 'drop_zone' / p.name)
for p in pos_paths[split_pos:]:
    shutil.copy(p, DETECT_ROOT / 'val' / 'drop_zone' / p.name)

neg_paths = list(not_dz_paths)
random.shuffle(neg_paths)
split_neg = max(1, int(len(neg_paths) * 0.85))
for p in neg_paths[:split_neg]:
    shutil.copy(p, DETECT_ROOT / 'train' / 'not_drop_zone' / p.name)
for p in neg_paths[split_neg:]:
    shutil.copy(p, DETECT_ROOT / 'val' / 'not_drop_zone' / p.name)

for split in ('train', 'val'):
    counts = {c: len(list((DETECT_ROOT/split/c).iterdir()))
              for c in ('drop_zone', 'not_drop_zone')}
    print(f'{split}: {counts}')

In [ ]:
# Cell 4 — Train Stage 1 detector
BATCH = 16

det_train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.RandomRotation(20),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

det_train_ds = ImageFolder(str(DETECT_ROOT / 'train'), transform=det_train_tf)
det_val_ds   = ImageFolder(str(DETECT_ROOT / 'val'),   transform=INFERENCE_TF)

counts_det = Counter(det_train_ds.targets)
weights    = [1.0 / counts_det[t] for t in det_train_ds.targets]
sampler    = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

det_train_loader = DataLoader(det_train_ds, batch_size=BATCH, sampler=sampler,  num_workers=2)
det_val_loader   = DataLoader(det_val_ds,   batch_size=BATCH, shuffle=False,    num_workers=2)

print('class_to_idx:', det_train_ds.class_to_idx)
DZ_IDX = det_train_ds.class_to_idx['drop_zone']
print(f'drop_zone class index: {DZ_IDX}')

# Build model
def make_efficientnet(num_classes: int):
    net = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    for param in net.parameters():
        param.requires_grad = False
    in_features = net.classifier[1].in_features
    net.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_features, num_classes))
    return net.to(DEVICE)

def train_model(model, train_loader, val_loader, epochs, lr, unfreeze_blocks=None):
    if unfreeze_blocks:
        for name, param in model.named_parameters():
            if any(b in name for b in unfreeze_blocks) or 'classifier' in name:
                param.requires_grad = True
    params    = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = optim.Adam(params, lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    best_acc, best_state = 0, None
    for epoch in range(1, epochs + 1):
        model.train()
        tr_correct, tr_total = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            tr_correct += (out.argmax(1) == labels).sum().item()
            tr_total   += len(imgs)
        model.eval()
        vl_correct, vl_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                vl_correct += (model(imgs).argmax(1) == labels).sum().item()
                vl_total   += len(imgs)
        vl_acc = vl_correct / max(1, vl_total)
        scheduler.step()
        flag = ''
        if vl_acc > best_acc:
            best_acc   = vl_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            flag = '  ← saved'
        print(f'Ep {epoch:2d} | train={tr_correct/max(1,tr_total):.3f} | val={vl_acc:.3f}{flag}')
    model.load_state_dict(best_state)
    return best_acc

print('\nBuilding Stage 1 detector (EfficientNet B0, binary)...')
detector = make_efficientnet(num_classes=2)

print('Phase 1 — head only (12 epochs)')
acc1 = train_model(detector, det_train_loader, det_val_loader, epochs=12, lr=1e-3)

print('\nPhase 2 — fine-tune last blocks (12 epochs)')
acc2 = train_model(detector, det_train_loader, det_val_loader, epochs=12, lr=1e-4,
                   unfreeze_blocks=['features.7', 'features.8'])

print(f'\nBest Stage 1 val acc: {max(acc1, acc2):.3f}')

In [ ]:
# Cell 5 — Evaluate and save Stage 1 detector
detector.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in det_val_loader:
        preds = detector(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=det_val_ds.classes))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=det_val_ds.classes, yticklabels=det_val_ds.classes, cmap='Greens')
plt.title('Stage 1 — Drop Zone Detector Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('dz_detector_confusion.png', dpi=150)
plt.show()

det_bundle = {
    'model_state_dict': detector.state_dict(),
    'class_to_idx':     det_train_ds.class_to_idx,
    'idx_to_class':     {v: k for k, v in det_train_ds.class_to_idx.items()},
    'dz_class_idx':     DZ_IDX,
    'architecture':     'efficientnet_b0',
    'input_size':       IMG_SIZE,
    'val_acc':          round(max(acc1, acc2), 4),
    'version':          1,
}
torch.save(det_bundle, 'drop_zone_detector.pt')
shutil.copy('drop_zone_detector.pt', MODEL_OUT / 'drop_zone_detector.pt')
print(f'Saved: drop_zone_detector.pt  (val_acc={det_bundle["val_acc"]})')

In [ ]:
# Cell 6 — Stage 2 choice
# ─────────────────────────────────────────────────────────────────────────────
# OPTION A (default): Supervised EfficientNet B0 binary classifier
#   Requires: 30+ labeled bad photos
#   Pros: learns BOTH good and bad explicitly — far more accurate than IF
#   Output: drop_zone_quality.pt
#
# OPTION B: DINOv2 + Isolation Forest (one-class)
#   Use only if you have fewer than 30 bad photos
#   Pros: works with very few bad examples
#   Cons: only learns "good" — misses defects that look visually similar to good
#   Output: drop_zone_model.pkl
# ─────────────────────────────────────────────────────────────────────────────

USE_SUPERVISED = len(bad_paths) >= 30   # auto-select based on bad photo count

print(f'Bad photos available : {len(bad_paths)}')
print(f'Stage 2 mode         : {"Supervised EfficientNet B0" if USE_SUPERVISED else "DINOv2 + Isolation Forest"}')
if not USE_SUPERVISED:
    print('  (add more bad photos to reach 30+ and unlock supervised mode)')

In [ ]:
# Cell 7 — Train Stage 2 quality model
import shutil, random
from collections import Counter
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision.datasets import ImageFolder

if USE_SUPERVISED:
    # ── Supervised EfficientNet B0 (good vs defect) ───────────────────────────
    # Augment bad photos to match good count
    aug_tf = T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3),
        T.RandomRotation(25),
        T.RandomPerspective(distortion_scale=0.3, p=0.4),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    ])

    QUAL_ROOT = Path('dz_quality_split')
    for split in ('train', 'val'):
        for cls in ('good', 'defect'):
            (QUAL_ROOT / split / cls).mkdir(parents=True, exist_ok=True)

    random.seed(42)

    # Copy good photos
    good_shuf = list(good_paths); random.shuffle(good_shuf)
    n_good_train = max(1, int(len(good_shuf) * 0.85))
    for p in good_shuf[:n_good_train]: shutil.copy(p, QUAL_ROOT / 'train' / 'good' / p.name)
    for p in good_shuf[n_good_train:]: shutil.copy(p, QUAL_ROOT / 'val'   / 'good' / p.name)

    # Copy bad photos + augment to match good count
    bad_shuf = list(bad_paths); random.shuffle(bad_shuf)
    n_bad_train = max(1, int(len(bad_shuf) * 0.85))
    for p in bad_shuf[:n_bad_train]: shutil.copy(p, QUAL_ROOT / 'train' / 'defect' / p.name)
    for p in bad_shuf[n_bad_train:]: shutil.copy(p, QUAL_ROOT / 'val'   / 'defect' / p.name)

    # Augment defect train to balance with good
    needed = n_good_train - n_bad_train
    if needed > 0:
        aug_per = max(1, needed // max(1, n_bad_train)) + 1
        added = 0
        for orig in bad_shuf[:n_bad_train]:
            img = Image.open(orig).convert('RGB')
            for j in range(aug_per):
                out = QUAL_ROOT / 'train' / 'defect' / f'{orig.stem}_aug{j:03d}{orig.suffix}'
                aug_tf(img).save(out)
                added += 1
                if added >= needed: break
            if added >= needed: break
        print(f'Augmented defect train: {n_bad_train} originals + {added} augmented')

    qual_train_tf = T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        T.RandomRotation(15),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    qual_train_ds = ImageFolder(str(QUAL_ROOT / 'train'), transform=qual_train_tf)
    qual_val_ds   = ImageFolder(str(QUAL_ROOT / 'val'),   transform=INFERENCE_TF)

    counts_q = Counter(qual_train_ds.targets)
    wts = [1.0 / counts_q[t] for t in qual_train_ds.targets]
    sampler_q = WeightedRandomSampler(wts, num_samples=len(wts), replacement=True)

    qual_train_loader = DataLoader(qual_train_ds, batch_size=16, sampler=sampler_q, num_workers=2)
    qual_val_loader   = DataLoader(qual_val_ds,   batch_size=16, shuffle=False,     num_workers=2)

    print('class_to_idx:', qual_train_ds.class_to_idx)
    DEFECT_IDX = qual_train_ds.class_to_idx['defect']

    print('\nTraining supervised quality model (good vs defect)...')
    quality_net = make_efficientnet(num_classes=2)

    print('Phase 1 — head only (12 epochs)')
    qacc1 = train_model(quality_net, qual_train_loader, qual_val_loader, epochs=12, lr=1e-3)
    print('\nPhase 2 — fine-tune last blocks (15 epochs)')
    qacc2 = train_model(quality_net, qual_train_loader, qual_val_loader, epochs=15, lr=5e-5,
                        unfreeze_blocks=['features.6', 'features.7', 'features.8'])
    print(f'\nBest quality val acc: {max(qacc1, qacc2):.3f}')

else:
    # ── DINOv2 + Isolation Forest fallback ────────────────────────────────────
    print('Loading DINOv2-small for Isolation Forest training...')
    dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', verbose=False)
    dino.eval().to(DEVICE)

    def extract_emb(path):
        try:
            img = Image.open(path).convert('RGB')
            t = DINO_TF(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                return dino(t).squeeze().cpu().numpy()
        except: return None

    X_good = np.array([e for p in good_paths if (e := extract_emb(p)) is not None])
    X_bad  = np.array([e for p in bad_paths  if (e := extract_emb(p)) is not None])
    print(f'Good embeddings: {len(X_good)}  Bad embeddings: {len(X_bad)}')

In [ ]:
# Cell 8 — Calibrate threshold
if USE_SUPERVISED:
    # ── Supervised threshold calibration ─────────────────────────────────────
    # Score every val photo to find a threshold that separates good from defect.
    # We use the midpoint between the median good-defect-prob and median bad-defect-prob.
    quality_net.eval()
    defect_probs_good = []
    defect_probs_bad  = []

    good_idx_in_ds = qual_val_ds.class_to_idx['good']

    with torch.no_grad():
        for imgs, labels in qual_val_loader:
            probs = torch.softmax(quality_net(imgs.to(DEVICE)), dim=1)[:, DEFECT_IDX].cpu().numpy()
            for p, l in zip(probs, labels.numpy()):
                if l == good_idx_in_ds:
                    defect_probs_good.append(p)
                else:
                    defect_probs_bad.append(p)

    median_good = float(np.median(defect_probs_good))
    median_bad  = float(np.median(defect_probs_bad))
    threshold   = round((median_good + median_bad) / 2, 4)

    print(f'Good defect-prob (median) : {median_good:.4f}')
    print(f'Bad  defect-prob (median) : {median_bad:.4f}')
    print(f'Calibrated threshold      : {threshold:.4f}  (midpoint)')
    print(f'  Defects caught  : {sum(1 for p in defect_probs_bad  if p >= threshold)}/{len(defect_probs_bad)}')
    print(f'  False positives : {sum(1 for p in defect_probs_good if p >= threshold)}/{len(defect_probs_good)}')
    print()
    print('Tip: lower threshold → catch more defects (more false positives).')
    print('     raise threshold → fewer false positives (might miss subtle defects).')

    # Score distribution
    plt.figure(figsize=(10, 5))
    plt.hist(defect_probs_good, bins=30, alpha=0.6, color='green', label=f'Good ({len(defect_probs_good)})')
    plt.hist(defect_probs_bad,  bins=20, alpha=0.7, color='red',   label=f'Bad ({len(defect_probs_bad)})')
    plt.axvline(threshold,   color='black',  linestyle='--', linewidth=2, label=f'Threshold = {threshold:.4f}')
    plt.axvline(median_good, color='green',  linestyle=':',  linewidth=1.5, label=f'Good median = {median_good:.4f}')
    plt.axvline(median_bad,  color='red',    linestyle=':',  linewidth=1.5, label=f'Bad median = {median_bad:.4f}')
    plt.xlabel('Defect Probability')
    plt.ylabel('Count')
    plt.title('Drop Zone Stage 2 (Supervised) — Score Distribution')
    plt.legend()
    plt.tight_layout()
    plt.savefig('dz_threshold.png', dpi=150)
    plt.show()

else:
    # ── Isolation Forest calibration ──────────────────────────────────────────
    print(f'Training Isolation Forest on {len(X_good)} good embeddings...')
    clf = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
    clf.fit(X_good)

    good_scores = clf.decision_function(X_good)
    bad_scores  = clf.decision_function(X_bad)

    print(f'\nGood score range : {good_scores.min():.4f} \u2013 {good_scores.max():.4f}')
    print(f'Bad  score range : {bad_scores.min():.4f}  \u2013 {bad_scores.max():.4f}')

    print('\nBad photo scores:')
    for path, score in sorted(zip(bad_paths, bad_scores), key=lambda x: x[1]):
        flag = 'FLAGGED' if score < 0 else 'MISSED'
        print(f'  {score:.4f}  [{flag}]  {path.name}')

    bad_median = float(np.median(bad_scores))
    bad_worst  = float(bad_scores.min())
    threshold  = bad_median * 1.05

    print(f'\nBad score median : {bad_median:.4f}')
    print(f'Bad score worst  : {bad_worst:.4f}')
    print(f'Calibrated threshold : {threshold:.4f}  (median-based)')
    print(f'  Bad caught   : {sum(1 for s in bad_scores  if s < threshold)}/{len(bad_scores)}')
    print(f'  Good flagged : {sum(1 for s in good_scores if s < threshold)}/{len(good_scores)}  (false positives)')
    print()
    print('Tip: if too many good photos are flagged, raise threshold toward bad_worst.')
    print('     if too many bad photos are missed, lower multiplier from 1.05 toward 1.0.')

    plt.figure(figsize=(10, 5))
    plt.hist(good_scores, bins=40, alpha=0.6, color='green', label=f'Good ({len(good_scores)})')
    plt.hist(bad_scores,  bins=20, alpha=0.7, color='red',   label=f'Bad ({len(bad_scores)})')
    plt.axvline(threshold, color='black',  linestyle='--', linewidth=2, label=f'Threshold (median) = {threshold:.4f}')
    plt.axvline(bad_worst, color='orange', linestyle=':',  linewidth=1.5, label=f'Worst bad = {bad_worst:.4f}')
    plt.axvline(0,         color='gray',   linestyle=':',  linewidth=1,   label='IF boundary (0)')
    plt.xlabel('Isolation Forest Score')
    plt.ylabel('Count')
    plt.title('Drop Zone Stage 2 (IF) \u2014 Score Distribution')
    plt.legend()
    plt.tight_layout()
    plt.savefig('dz_threshold.png', dpi=150)
    plt.show()


In [ ]:
# Cell 9 — Save Stage 2 model + download both models
if USE_SUPERVISED:
    quality_bundle = {
        'model_state_dict': quality_net.state_dict(),
        'class_to_idx':     qual_train_ds.class_to_idx,
        'idx_to_class':     {v: k for k, v in qual_train_ds.class_to_idx.items()},
        'defect_class_idx': DEFECT_IDX,
        'architecture':     'efficientnet_b0',
        'input_size':       IMG_SIZE,
        'threshold':        threshold,
        'good_count':       len(good_paths),
        'bad_count':        len(bad_paths),
        'val_acc':          round(max(qacc1, qacc2), 4),
        'version':          1,
    }
    quality_path = 'drop_zone_quality.pt'
    torch.save(quality_bundle, quality_path)
    shutil.copy(quality_path,       MODEL_OUT / quality_path)
    shutil.copy('dz_threshold.png', MODEL_OUT / 'dz_threshold.png')
    print(f'Stage 2 [supervised] saved: {quality_path}')
    print(f'  good_count    : {quality_bundle["good_count"]}')
    print(f'  bad_count     : {quality_bundle["bad_count"]}')
    print(f'  val_acc       : {quality_bundle["val_acc"]}')
    print(f'  threshold     : {quality_bundle["threshold"]:.4f}')
    print(f'  defect_idx    : {DEFECT_IDX}')
else:
    quality_bundle = {
        'clf':              clf,
        'threshold':        threshold,
        'good_count':       len(X_good),
        'bad_count':        len(X_bad),
        'dinov2_model':     'dinov2_vits14',
        'good_score_range': (float(good_scores.min()), float(good_scores.max())),
        'bad_score_range':  (float(bad_scores.min()),  float(bad_scores.max())),
    }
    quality_path = 'drop_zone_model.pkl'
    with open(quality_path, 'wb') as f:
        pickle.dump(quality_bundle, f)
    shutil.copy(quality_path,       MODEL_OUT / quality_path)
    shutil.copy('dz_threshold.png', MODEL_OUT / 'dz_threshold.png')
    print(f'Stage 2 [isolation_forest] saved: {quality_path}')
    print(f'  good_count : {quality_bundle["good_count"]}')
    print(f'  bad_count  : {quality_bundle["bad_count"]}')
    print(f'  threshold  : {quality_bundle["threshold"]:.4f}')

print('\n' + '='*60)
print('Downloading both models...')
from google.colab import files
files.download('drop_zone_detector.pt')
files.download(quality_path)
files.download('dz_threshold.png')

print('\nNext steps:')
print(f'  1. Drop drop_zone_detector.pt into  drop_zone-model/')
if USE_SUPERVISED:
    print(f'  2. Drop drop_zone_quality.pt  into  drop_zone-model/')
else:
    print(f'  2. Drop drop_zone_model.pkl   into  drop_zone-model/')
print(f'  3. Run: venv/bin/python3 run_drop_zone_report.py')
